# fp16 LoRA: Training Apple's 3B Model on a Free T4

Three patches to Apple's adapter training toolkit enable training on Colab's free T4 GPU (16GB):

1. **mmap loading** — reads weights from disk on demand, avoids 12GB system RAM spike
2. **fp16 model + fp32 adapters** — halves GPU memory from 12GB to 6GB
3. **rms_norm fix + gradient scaling** — fixes dtype mismatches that cause NaN

Result: ~2 hours training on free T4. This is half-precision LoRA (fp16 base), not true QLoRA (4-bit NF4).

## 1. Setup

Upload to `My Drive/hunch-training/`:
- `adapter_training_toolkit_v26_0_0/` (from developer.apple.com)
- `prepare_data.py`, `tldr_bank.db`, `prompts.jsonl`

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

DRIVE_DIR = '/content/drive/MyDrive/hunch-training'
WORK_DIR = '/content/hunch-training'

!mkdir -p {WORK_DIR}
!cp -r {DRIVE_DIR}/adapter_training_toolkit_v26_0_0 {WORK_DIR}/
!cp {DRIVE_DIR}/prepare_data.py {WORK_DIR}/
!mkdir -p {WORK_DIR}/../bank {WORK_DIR}/../benchmark
!cp {DRIVE_DIR}/tldr_bank.db {WORK_DIR}/../bank/
!cp {DRIVE_DIR}/prompts.jsonl {WORK_DIR}/../benchmark/

!cd {WORK_DIR}/adapter_training_toolkit_v26_0_0 && pip install -r requirements.txt -q

import torch
print(f'CUDA: {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')

## 2. Prepare training data

In [ ]:
!cd {WORK_DIR} && python3 prepare_data.py

## 3. Apply patches

Three patches make training fit on T4 (16GB GPU, 12GB RAM):

**Patch 1 — `utils.py`:** mmap loading (0 RAM), fp16 model (6GB GPU), fp32 adapters (stable gradients)

**Patch 2 — `train_adapter.py`:** enable gradient scaling for f16-mixed (prevents NaN overflow)

**Patch 3 — `tamm/layers/functional.py`:** cast rms_norm weight to match input dtype (prevents NaN from dtype mismatch)

In [ ]:
import glob, shutil

# --- Patch 1: utils.py ---
# Restore clean copy first
!cp {DRIVE_DIR}/adapter_training_toolkit_v26_0_0/examples/utils.py \
    {WORK_DIR}/adapter_training_toolkit_v26_0_0/examples/utils.py

utils_path = f'{WORK_DIR}/adapter_training_toolkit_v26_0_0/examples/utils.py'
code = open(utils_path).read()

# 1a: Force fp16 model creation (6GB instead of 12GB on GPU)
code = code.replace(
    'model_config.dtype = dtype or model_config.dtype',
    'model_config.dtype = torch.float16'
)

# 1b: mmap loading (weights stay on disk, ~0 system RAM)
code = code.replace(
    '''    with Path(base_model_checkpoint_path).open("rb") as f:\n        sd = torch.load(f, map_location=device, weights_only=False)\n        _ = model.load_state_dict(sd, strict=True)''',
    '''    sd = torch.load(str(base_model_checkpoint_path), map_location=device, mmap=True, weights_only=False)\n    _ = model.load_state_dict(sd, strict=True)\n    del sd; import gc; gc.collect()'''
)

# 1c: Keep adapter weights in fp32 (GradScaler needs fp32 gradients)
code = code.replace(
    '    return model.to(device=device, dtype=model_config.dtype)',
    '''    model = model.to(device=device, dtype=model_config.dtype)

    # Keep adapter weights in fp32 for stable training
    for name, parameter in model.named_parameters():
        if "adapter" in name:
            parameter.data = parameter.data.float()

    return model'''
)

open(utils_path, 'w').write(code)
print('Patch 1 applied: utils.py (mmap + fp16 + fp32 adapters)')

# --- Patch 2: train_adapter.py ---
!cp {DRIVE_DIR}/adapter_training_toolkit_v26_0_0/examples/train_adapter.py \
    {WORK_DIR}/adapter_training_toolkit_v26_0_0/examples/train_adapter.py

ta_path = f'{WORK_DIR}/adapter_training_toolkit_v26_0_0/examples/train_adapter.py'
code = open(ta_path).read()
code = code.replace(
    'return self.precision == "f16"',
    'return self.precision in ("f16", "f16-mixed")'
)
open(ta_path, 'w').write(code)
print('Patch 2 applied: train_adapter.py (gradient scaling for f16-mixed)')

# --- Patch 3: tamm rms_norm ---
norm_files = glob.glob(f'{WORK_DIR}/**/tamm/layers/functional.py', recursive=True)
norm_files += glob.glob('/usr/local/lib/**/tamm/layers/functional.py', recursive=True)
for nf in norm_files:
    code = open(nf).read()
    if 'weight.to(tensor.dtype)' not in code:
        old = '        tensor = _torch_compatibility.rms_norm(\n            tensor, normalized_shape=normalized_shape, weight=weight, eps=eps\n        )'
        new = '        if weight is not None and weight.dtype != tensor.dtype:\n            weight = weight.to(tensor.dtype)\n        tensor = _torch_compatibility.rms_norm(\n            tensor, normalized_shape=normalized_shape, weight=weight, eps=eps\n        )'
        code = code.replace(old, new)
        open(nf, 'w').write(code)
        print(f'Patch 3 applied: {nf} (rms_norm dtype fix)')
    else:
        print(f'Patch 3 already applied: {nf}')

# Clear pycache
for d in glob.glob(f'{WORK_DIR}/**/tamm/**/__pycache__', recursive=True):
    shutil.rmtree(d, ignore_errors=True)
for d in glob.glob('/usr/local/lib/**/tamm/**/__pycache__', recursive=True):
    shutil.rmtree(d, ignore_errors=True)
print('\nAll patches applied. Ready to train.')

## 4. Train

~40 min/epoch on T4, ~2 hours total for 3 epochs.

In [ ]:
!cd {WORK_DIR}/adapter_training_toolkit_v26_0_0 && python3 -m examples.train_adapter \
  --train-data ../train.jsonl \
  --eval-data ../eval.jsonl \
  --epochs 3 \
  --learning-rate 1e-4 \
  --batch-size 8 \
  --precision f16-mixed \
  --activation-checkpointing \
  --checkpoint-dir ../fp16-lora-checkpoints/

## 5. Save checkpoints

In [ ]:
!cp -r {WORK_DIR}/fp16-lora-checkpoints {DRIVE_DIR}/
!echo 'Checkpoints saved to Drive'

## 6. Evaluate

In [ ]:
import json, subprocess

test_prompts = [
    'find files changed in the last hour',
    'show disk usage',
    'generate a random password',
    'kill a process by name',
    'show http headers of a url',
    'record terminal session',
    'find files larger than 100mb',
    'convert image to different format',
    'show all listening ports',
    'find files modified in the last 7 days',
    'find files owned by root',
    'count lines in all python files',
    'show all environment variables',
    'clear the terminal',
    'compare two files',
]

system = 'Output a single shell command for zsh on macOS. No explanation, no markdown, no backticks. Just the command.'

with open(f'{WORK_DIR}/test_prompts.jsonl', 'w') as f:
    for p in test_prompts:
        f.write(json.dumps([
            {'role': 'system', 'content': system},
            {'role': 'user', 'content': p}
        ]) + '\n')

result = subprocess.run(
    ['python3', '-m', 'examples.generate',
     '--prompt', '../test_prompts.jsonl',
     '--checkpoint', '../fp16-lora-checkpoints/adapter-final.pt',
     '--precision', 'f16-mixed'],
    capture_output=True, text=True,
    cwd=f'{WORK_DIR}/adapter_training_toolkit_v26_0_0'
)

lines = (result.stdout + result.stderr).strip().split('\n')
idx = 0
for line in lines:
    if 'Response for prompt' in line:
        answer = line.split(': ', 2)[-1].replace('<turn_end>', '').strip()
        prompt = test_prompts[idx] if idx < len(test_prompts) else '?'
        print(f'Q: {prompt:<45} A: {answer}')
        idx += 1

if idx == 0:
    print('No output. Check error:')
    print('STDERR:', result.stderr[-500:])
    print('Return code:', result.returncode)

## 7. Export .fmadapter

In [ ]:
!cd {WORK_DIR}/adapter_training_toolkit_v26_0_0 && python3 -m export.export_fmadapter \
  --adapter-name hunch_fp16 \
  --checkpoint ../fp16-lora-checkpoints/adapter-final.pt \
  --output-dir ../fp16-lora-exports/

!ls -lh {WORK_DIR}/fp16-lora-exports/
!cp -r {WORK_DIR}/fp16-lora-exports {DRIVE_DIR}/
!echo 'Adapter exported and saved to Drive'